# dim_team Backfill Notebook

**Issue #173** — Populate the `team` table with records for every tri-code
referenced in `live_game.away_code` or `live_game.home_code` that is not
already present in the `team` table.

Fetches team data from the NHL Stats REST API at
`https://api.nhle.com/stats/rest/en/team` (team list + IDs) and
`https://api.nhle.com/stats/rest/en/team/id/{id}` (per-team detail).

## Run instructions

```bash
pip install jupyter pandas httpx sqlalchemy
jupyter notebook nhl-dashboard/notebooks/dim_team_backfill.ipynb
```

Run all cells top-to-bottom.

## Notebook structure

| Section | Content |
|---|---|
| Setup | Imports, SQLAlchemy engine pointing to `instance/nhl.db`, NHL Stats API base URL |
| Section 1 — Detect missing teams | Query `live_game` for all `away_code`/`home_code` values; subtract codes already in `team` |
| Section 2 — Resolve team IDs | Call `GET /stats/rest/en/team` to build a `triCode → id` lookup map |
| Section 3 — Fetch per-team detail | Call `GET /stats/rest/en/team/id/{id}` for each missing code |
| Section 4 — Upsert function | `upsert_team(session, team_dict)` using `session.merge()` on `tri_code` PK |
| Section 5 — Batch upsert | Loop over missing teams, upsert, commit once per batch |
| Section 6 — Verification | Before/after row counts, sample of newly inserted rows |

## Setup

Imports, SQLAlchemy connection to `instance/nhl.db`, and the NHL Stats REST API
base URL. No Flask app context — models are imported directly from
`../backend/models.py`. `httpx` is used for all HTTP calls with 50 ms
rate-limiting between requests.

In [ ]:
import sys
import time
from pathlib import Path

import httpx
import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.orm import sessionmaker

# Add backend to path so we can import models without Flask app context
BACKEND_DIR = Path("../backend").resolve()
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

NHL_STATS_BASE = "https://api.nhle.com/stats/rest/en"
DB_PATH = BACKEND_DIR / "instance" / "nhl.db"

# Connect directly to the Flask app's SQLite DB — no app context needed
engine = create_engine(f"sqlite:///{DB_PATH}", echo=False)
Session = sessionmaker(bind=engine)

print(f"Database  : {DB_PATH}")
print(f"DB exists : {DB_PATH.exists()}")

## Section 1 — Detect missing teams

Queries `live_game` to collect every distinct tri-code referenced in
`away_code` or `home_code`, then subtracts tri-codes already present in the
`team` table to produce the list of `MISSING_CODES` to backfill.

In [ ]:
with engine.connect() as conn:
    # Union both columns so we get every tri-code referenced in live_game
    referenced_rows = conn.execute(text(
        "SELECT DISTINCT away_code AS tri_code FROM live_game WHERE away_code IS NOT NULL "
        "UNION "
        "SELECT DISTINCT home_code AS tri_code FROM live_game WHERE home_code IS NOT NULL"
    )).fetchall()

    existing_rows = conn.execute(text(
        "SELECT tri_code FROM team"
    )).fetchall()

referenced_codes = {row[0] for row in referenced_rows}
existing_codes   = {row[0] for row in existing_rows}
MISSING_CODES    = sorted(referenced_codes - existing_codes)

print(f"Tri-codes referenced in live_game : {len(referenced_codes)}")
print(f"Tri-codes already in team table   : {len(existing_codes)}")
print(f"Missing tri-codes to backfill     : {len(MISSING_CODES)}")
print(f"Missing : {MISSING_CODES}")

## Section 2 — Resolve team IDs

Calls `GET https://api.nhle.com/stats/rest/en/team` to retrieve the full list
of NHL teams with their numeric IDs and tri-codes. Builds a
`triCode → id` lookup map (`TRICODE_TO_ID`) used in Section 3 to look up the
correct ID for each missing code before fetching per-team detail.

In [ ]:
r = httpx.get(f"{NHL_STATS_BASE}/team", timeout=30)
r.raise_for_status()
team_list_data = r.json()

# Build triCode → id lookup map from the API response
TRICODE_TO_ID = {}
for team_entry in team_list_data.get("data", []):
    tri_code = team_entry.get("triCode")
    team_id  = team_entry.get("id")
    if tri_code and team_id:
        TRICODE_TO_ID[tri_code] = team_id

print(f"Teams in API response     : {len(TRICODE_TO_ID)}")
print(f"Missing codes resolvable  : {sum(1 for c in MISSING_CODES if c in TRICODE_TO_ID)}")
unresolvable = [c for c in MISSING_CODES if c not in TRICODE_TO_ID]
if unresolvable:
    print(f"WARNING — unresolvable codes (not in API): {unresolvable}")

## Section 3 — Fetch per-team detail

For each missing tri-code that can be resolved to a numeric team ID, calls
`GET https://api.nhle.com/stats/rest/en/team/id/{id}` to retrieve the full
team record including `franchiseId`, `fullName`, `leagueId`, and `rawTricode`.

Stores results in `TEAM_DETAIL_MAP` keyed by tri-code. Rate-limited at 50 ms
between requests.

In [ ]:
TEAM_DETAIL_MAP = {}  # tri_code → detail dict from /stats/rest/en/team/id/{id}

for tri_code in MISSING_CODES:
    team_id = TRICODE_TO_ID.get(tri_code)
    if team_id is None:
        print(f"  SKIP {tri_code}: no numeric ID found in team list")
        continue

    try:
        r = httpx.get(f"{NHL_STATS_BASE}/team/id/{team_id}", timeout=15)
        r.raise_for_status()
        detail = r.json()
        # Extract first record from data array
        records = detail.get("data", [])
        if records:
            TEAM_DETAIL_MAP[tri_code] = records[0]
            print(f"  OK {tri_code} (id={team_id}): {records[0].get('fullName')}")
        else:
            print(f"  SKIP {tri_code} (id={team_id}): empty data array")
    except Exception as exc:
        print(f"  FAIL {tri_code} (id={team_id}): {exc}")
        continue

    time.sleep(0.05)  # polite rate-limiting — 50 ms between requests

print()
print(f"Teams with detail fetched: {len(TEAM_DETAIL_MAP)} of {len(MISSING_CODES)} missing")

## Section 4 — Upsert function

Defines `upsert_team(session, team_dict)` which:

1. Extracts all `team` table columns from a per-team detail dict
2. Constructs a `Team` instance with `tri_code`, `team_id`, `franchise_id`,
   `full_name`, `league_id`, `raw_tricode`, and `name` (alias for `fullName`)
3. Calls `session.merge()` to upsert by `tri_code` primary key — idempotent
   on repeated runs

In [ ]:
# Import the SQLAlchemy model — keeps column definitions in sync with live schema
from models import Team


def upsert_team(session, team_dict: dict) -> None:
    """Upsert one row into the team table from a /stats/rest/en/team/id/{id} response.

    Uses session.merge() so the call is idempotent: INSERT on first run,
    UPDATE on subsequent runs. tri_code is the primary key.

    Args:
        session: SQLAlchemy Session bound to instance/nhl.db.
        team_dict: Raw team object from /stats/rest/en/team/id/{id} data array.
    """
    full_name  = team_dict.get("fullName")
    raw_tri    = team_dict.get("rawTricode")
    tri_code   = team_dict.get("triCode") or raw_tri

    row = Team(
        tri_code     = tri_code,
        name         = full_name,
        team_id      = team_dict.get("id"),
        franchise_id = team_dict.get("franchiseId"),
        full_name    = full_name,
        league_id    = team_dict.get("leagueId"),
        raw_tricode  = raw_tri,
    )
    session.merge(row)


print("upsert_team() defined")
print("Columns populated:")
for col in ["tri_code", "name", "team_id", "franchise_id", "full_name", "league_id", "raw_tricode"]:
    print(f"  {col}")

## Section 5 — Batch upsert

Loops over all entries in `TEAM_DETAIL_MAP` and calls `upsert_team()` for each.
Commits once after the full batch — the dataset is small (at most a handful of
missing teams), so per-row commits are unnecessary. Individual failures are
logged and skipped without aborting the loop.

In [ ]:
upserted = 0
failed   = 0

with Session() as session:
    for tri_code, detail in TEAM_DETAIL_MAP.items():
        try:
            upsert_team(session, detail)
            upserted += 1
            print(f"  OK  {tri_code}: {detail.get('fullName')}")
        except Exception as exc:
            print(f"  FAIL {tri_code}: {exc}")
            failed += 1

    session.commit()  # commit once after the batch

print()
print(f"Batch complete — upserted: {upserted}, failed: {failed}")

## Section 6 — Verification

Queries the `team` table to confirm the backfill succeeded:

1. **Before/after row counts** — compares `COUNT_BEFORE` (captured before
   Section 5) with the current count to confirm new rows were inserted
2. **Sample of newly inserted rows** — displays a DataFrame of the teams that
   were just upserted, keyed by `MISSING_CODES`

In [ ]:
# Capture the before-count (run this cell once before Section 5 if you want the
# comparison; on a fresh run it will equal the after-count if run after Section 5)
with engine.connect() as conn:
    COUNT_BEFORE = conn.execute(text("SELECT COUNT(*) FROM team")).scalar()

print(f"team row count before backfill : {COUNT_BEFORE}")

In [ ]:
with engine.connect() as conn:
    count_after = conn.execute(text("SELECT COUNT(*) FROM team")).scalar()

    # Sample of newly upserted teams
    if MISSING_CODES:
        placeholders = ",".join(f"'{c}'" for c in MISSING_CODES)
        sample_sql   = f"""
            SELECT tri_code, name, team_id, franchise_id, full_name, league_id, raw_tricode
            FROM team
            WHERE tri_code IN ({placeholders})
            ORDER BY tri_code
        """
        sample_rows = conn.execute(text(sample_sql)).fetchall()
    else:
        sample_rows = []

print(f"team row count before backfill : {COUNT_BEFORE}")
print(f"team row count after  backfill : {count_after}")
print(f"Net new rows inserted          : {count_after - COUNT_BEFORE}")
print()

if sample_rows:
    df_sample = pd.DataFrame(
        sample_rows,
        columns=["tri_code", "name", "team_id", "franchise_id", "full_name", "league_id", "raw_tricode"],
    )
    print("Newly upserted teams:")
    display(df_sample)
else:
    print("No missing teams detected — team table is already complete.")